# 02 - Analise e limpeza dos dados

Este notebook analisa `dataset_completo.csv`, gerado pelo `01_MegaJoin.ipynb`, e cria uma copia limpa. O arquivo bruto nao e alterado.

A limpeza desta etapa remove apenas colunas 100% nulas e colunas constantes. Nao preenche nulos nem remove linhas.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option('display.max_columns', 30)
pd.set_option('display.max_rows', 40)
pd.set_option('display.width', 160)

DATA_PATH = Path('dataset_completo.csv')
df = pd.read_csv(DATA_PATH, low_memory=False)
print(f'Arquivo: {DATA_PATH}')
print(f'Dimensoes: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas')

Arquivo: dataset_completo.csv
Dimensoes: 9,146 linhas x 347 colunas


## Visao geral

In [2]:
resumo = {
    'linhas': len(df),
    'colunas': df.shape[1],
    'temporadas': f"{df['race_year'].min()}-{df['race_year'].max()}" if 'race_year' in df else 'coluna ausente',
    'corridas': df['rd_race_id'].nunique() if 'rd_race_id' in df else None,
    'pilotos': df['rd_driver_id'].nunique() if 'rd_driver_id' in df else None,
    'resultados vencedores': int(df['rd_position_number'].eq(1).sum()) if 'rd_position_number' in df else None,
}
pd.Series(resumo, name='valor')

linhas                        9146
colunas                        347
temporadas               2003-2026
corridas                       436
pilotos                        117
resultados vencedores          436
Name: valor, dtype: object

## Tipos de dados

In [3]:
tipos = (
    pd.DataFrame({
        'tipo': df.dtypes.astype(str),
        'valores distintos': df.nunique(dropna=True),
    })
    .groupby('tipo')
    .agg(colunas=('tipo', 'size'), valores_distintos_mediana=('valores distintos', 'median'))
    .sort_values('colunas', ascending=False)
)
tipos

,colunas,valores_distintos_mediana
tipo,,
float64,147,8.0
str,129,30.0
int64,71,12.0


## Valores ausentes

In [4]:
nulos = df.isna().sum().rename('nulos').to_frame()
nulos['percentual'] = (nulos['nulos'] / len(df) * 100).round(2)
nulos = nulos.sort_values(['nulos', 'percentual'], ascending=False)

print(f"Colunas com ao menos um nulo: {(nulos['nulos'] > 0).sum()} de {df.shape[1]}")
print(f"Colunas totalmente vazias: {(nulos['nulos'] == len(df)).sum()}")
print(f"Linhas sem nenhum nulo: {df.notna().all(axis=1).sum():,} de {len(df):,}")
print('Colunas com mais nulos:')
display(nulos[nulos['nulos'] > 0].head(30))

colunas_vazias = nulos.index[nulos['nulos'] == len(df)].tolist()
print(f'\nColunas totalmente vazias ({len(colunas_vazias)}):')
relatorio_vazias = pd.DataFrame({
    'coluna': colunas_vazias,
    'valor': [df[coluna].iloc[0] for coluna in colunas_vazias],
})
display(relatorio_vazias)

colunas_95pct = nulos.index[nulos['percentual'] >= 95].tolist()
print(f'\nColunas com pelo menos 95% de nulos: {len(colunas_95pct)}')
print(colunas_95pct)

print('\nNulos por linha:')
display(df.isna().sum(axis=1).describe().to_frame('nulos_por_linha'))

Colunas com ao menos um nulo: 172 de 347
Colunas totalmente vazias: 60


Linhas sem nenhum nulo: 0 de 9,146
Colunas com mais nulos:


,nulos,percentual
rd_practice_time,9146,100.0
rd_practice_time_millis,9146,100.0
rd_practice_gap,9146,100.0
rd_practice_gap_millis,9146,100.0
rd_practice_interval,9146,100.0
rd_practice_interval_millis,9146,100.0
rd_practice_laps,9146,100.0
rd_qualifying_time,9146,100.0
rd_qualifying_time_millis,9146,100.0
rd_qualifying_q1,9146,100.0



Colunas totalmente vazias (60):


,coluna,valor
0,rd_practice_time,NaN
1,rd_practice_time_millis,NaN
2,rd_practice_gap,NaN
3,rd_practice_gap_millis,NaN
4,rd_practice_interval,NaN
...,...,...
55,race_warming_up_date,NaN
56,race_warming_up_time,NaN
57,sess_prequali_position_number,NaN
58,sess_prequali_position_text,NaN



Colunas com pelo menos 95% de nulos: 68
['rd_practice_time', 'rd_practice_time_millis', 'rd_practice_gap', 'rd_practice_gap_millis', 'rd_practice_interval', 'rd_practice_interval_millis', 'rd_practice_laps', 'rd_qualifying_time', 'rd_qualifying_time_millis', 'rd_qualifying_q1', 'rd_qualifying_q1_millis', 'rd_qualifying_q2', 'rd_qualifying_q2_millis', 'rd_qualifying_q3', 'rd_qualifying_q3_millis', 'rd_qualifying_gap', 'rd_qualifying_gap_millis', 'rd_qualifying_interval', 'rd_qualifying_interval_millis', 'rd_qualifying_laps', 'rd_starting_grid_position_qualification_position_number', 'rd_starting_grid_position_qualification_position_text', 'rd_starting_grid_position_grid_penalty', 'rd_starting_grid_position_grid_penalty_positions', 'rd_starting_grid_position_time', 'rd_starting_grid_position_time_millis', 'rd_fastest_lap_lap', 'rd_fastest_lap_time', 'rd_fastest_lap_time_millis', 'rd_fastest_lap_gap', 'rd_fastest_lap_gap_millis', 'rd_fastest_lap_interval', 'rd_fastest_lap_interval_millis

,nulos_por_linha
count,9146.000000
mean,101.000437
std,6.752842
min,83.000000
25%,97.000000
50%,101.000000
75%,105.000000
max,148.000000


### Decisao para as colunas 100% nulas

Vamos remover da copia limpa as colunas sem nenhum valor neste recorte. Os nulos se agrupam por estes motivos:

- `rd_practice_*`, `rd_qualifying_*`, `rd_starting_grid_*` e `rd_fastest_lap_*` pertencem a outros tipos de registro em `race_data`; a consulta principal usa `RACE_RESULT`. As sessoes selecionadas estao nas colunas `sess_*`.
- `rd_pit_stop_*` e `rd_driver_of_the_day_percentage` pertencem aos tipos `PIT_STOP` e `DRIVER_OF_THE_DAY_RESULT`, que nao entram como linhas neste join.
- `sess_prequali_*` fica vazio porque nao ha resultados de pre-classificacao a partir de 2003.
- `race_sprint_*` fica vazio depois de excluir todos os fins de semana sprint pelos campos de sprint preenchidos no banco.
- As datas de pre-classificacao, FP4, quali 1/2 e warm-up nao estao preenchidas no banco original.

## Colunas sem variacao

A contagem anterior juntava colunas vazias e colunas com um valor repetido: `nunique(dropna=False)` conta o nulo como um valor distinto. O codigo separa esses dois casos e mostra o valor das colunas constantes.

As constantes sao removidas porque nao distinguem linhas nesta amostra:

- `rd_type`: todas as linhas sao `RACE_RESULT`, por causa do filtro da consulta.
- `rd_race_shared_car`: vale `0` em todas as linhas; a amostra 2003+ nao tem carros compartilhados.
- `driver_gender`: vale `MALE` em todas as linhas; nao varia neste dataset.
- `tyremfr_best_starting_grid_position`: vale `1` em todas as linhas e nao separa os resultados.
- `tyremfr_best_race_result`: vale `1` em todas as linhas e nao separa os resultados.

O codigo tambem procura strings vazias e marcadores de ausencia, sem alterar dados.

In [5]:
valores_distintos = df.nunique(dropna=False)
colunas_vazias = df.columns[df.isna().all()].tolist()
colunas_constantes = [
    coluna for coluna in df.columns
    if valores_distintos[coluna] <= 1 and coluna not in colunas_vazias
]

print(f'Colunas totalmente vazias: {len(colunas_vazias)}')
print(f'Colunas constantes com valor: {len(colunas_constantes)}')

relatorio_constantes = pd.DataFrame({
    'valor unico': [df[coluna].iloc[0] for coluna in colunas_constantes],
    'tipo': [str(df[coluna].dtype) for coluna in colunas_constantes],
}, index=colunas_constantes)
display(relatorio_constantes)

colunas_texto = [
    coluna for coluna in df.columns
    if pd.api.types.is_string_dtype(df[coluna].dtype) or df[coluna].dtype == object
]
texto_vazio = {}
marcadores = {'NULL', 'N/A', 'NA', 'NAN', r'\N'}
marcadores_encontrados = {}
for coluna in colunas_texto:
    serie = df[coluna].dropna().astype('string').str.strip()
    vazios = int(serie.eq('').sum())
    if vazios:
        texto_vazio[coluna] = vazios
    contagem = serie[serie.str.upper().isin(marcadores)].value_counts()
    if not contagem.empty:
        marcadores_encontrados[coluna] = contagem.to_dict()

print(f'Colunas de texto com strings vazias: {len(texto_vazio)}')
print(texto_vazio)
print(f'Colunas de texto com marcadores de ausencia: {len(marcadores_encontrados)}')
print(marcadores_encontrados)

Colunas totalmente vazias: 60
Colunas constantes com valor: 5


,valor unico,tipo
rd_type,RACE_RESULT,str
rd_race_shared_car,0,int64
driver_gender,MALE,str
tyremfr_best_starting_grid_position,1,int64
tyremfr_best_race_result,1,int64


Colunas de texto com strings vazias: 0
{}
Colunas de texto com marcadores de ausencia: 0
{}


### Sessoes antigas e dados biograficos

Tambem vamos retirar estas colunas da copia limpa:

- `sess_warmup_*` so tem dados em 2003; `sess_fp4_*` e `sess_quali1_*`/`sess_quali2_*` so tem dados entre 2003 e 2005. Esses formatos nao aparecem no periodo recente usado para validacao e teste.
- `driver_date_of_death` e `driver_second_nationality_country_id` foram removidas por decisao de escopo. A data de falecimento e posterior a parte dos resultados; a segunda nacionalidade so esta preenchida para tres pilotos.

O dataset bruto permanece preservado.

## Limpeza das colunas

A copia limpa remove colunas 100% nulas, constantes, sessoes antigas e os dois campos biograficos acima. Mantem todas as linhas e os valores das demais colunas. Nao preenche nulos.

### Penalidades da corrida

`rd_race_time_penalty` e `rd_race_time_penalty_millis` ficam na copia limpa para conferir resultados oficiais e penalidades aplicadas. Como a penalidade e conhecida durante ou depois da corrida, essas colunas nao podem ser usadas como features de previsao pre-corrida. Nao vamos transformar nulos em zero nesta etapa.

### Calendario com cobertura parcial

`race_free_practice_2_date` e `race_time` estao preenchidos apenas em 44 corridas de 2024 a 2026; o banco nao traz esses horarios para as temporadas anteriores. `race_scheduled_laps` e `race_scheduled_distance` aparecem em 35 corridas espalhadas entre 2003 e 2025. Esses campos descrevem o calendario ou a distancia programada, mas a cobertura e irregular.

Vamos mante-los na copia de analise e deixar os nulos como estao. Na selecao de features, por enquanto, ficam de fora: os dados de treino nao cobrem os mesmos periodos que validacao e teste, entao o modelo poderia aprender a diferenca de cobertura por temporada em vez de um padrao da corrida.

In [6]:
colunas_nulas = df.columns[df.isna().all()].tolist()
cardinalidade = df.nunique(dropna=False)
colunas_constantes = [
    coluna for coluna in df.columns
    if cardinalidade[coluna] == 1 and coluna not in colunas_nulas
]
colunas_sessoes_antigas = [
    'sess_warmup_position_number', 'sess_warmup_position_text',
    'sess_warmup_time_millis', 'sess_warmup_laps',
    'sess_fp4_position_number', 'sess_fp4_position_text',
    'sess_fp4_time_millis', 'sess_fp4_laps',
    'sess_quali1_position_number', 'sess_quali1_position_text',
    'sess_quali1_time_millis', 'sess_quali1_laps',
    'sess_quali2_position_number', 'sess_quali2_position_text',
    'sess_quali2_time_millis', 'sess_quali2_laps',
]
colunas_biograficas = [
    'driver_date_of_death',
    'driver_second_nationality_country_id',
]
colunas_sessoes_antigas = [c for c in colunas_sessoes_antigas if c in df.columns]
colunas_biograficas = [c for c in colunas_biograficas if c in df.columns]
colunas_removidas = list(dict.fromkeys(
    colunas_nulas + colunas_constantes + colunas_sessoes_antigas + colunas_biograficas
))

df_limpo = df.drop(columns=colunas_removidas).copy()

print(f'Colunas removidas por estarem 100% nulas: {len(colunas_nulas)}')
print(f'Colunas removidas por serem constantes: {len(colunas_constantes)}')
print(f'Colunas de sessoes antigas removidas: {len(colunas_sessoes_antigas)}')
print(f'Colunas biograficas removidas: {len(colunas_biograficas)}')
print(f'Colunas removidas no total: {len(colunas_removidas)}')
print(f'Dimensoes antes: {df.shape[0]:,} linhas x {df.shape[1]:,} colunas')
print(f'Dimensoes depois: {df_limpo.shape[0]:,} linhas x {df_limpo.shape[1]:,} colunas')
print(f'Colunas totalmente nulas restantes: {int(df_limpo.isna().all().sum())}')
print(f'Colunas constantes restantes: {int((df_limpo.nunique(dropna=False) <= 1).sum())}')

csv_limpo = Path('dataset_limpo.csv')
pkl_limpo = Path('dataset_limpo.pkl')
df_limpo.to_csv(csv_limpo, index=False)
df_limpo.to_pickle(pkl_limpo)
print(f'\nArquivos salvos: {csv_limpo} e {pkl_limpo}')

Colunas removidas por estarem 100% nulas: 60
Colunas removidas por serem constantes: 5
Colunas de sessoes antigas removidas: 16
Colunas biograficas removidas: 2
Colunas removidas no total: 83
Dimensoes antes: 9,146 linhas x 347 colunas
Dimensoes depois: 9,146 linhas x 264 colunas
Colunas totalmente nulas restantes: 0


Colunas constantes restantes: 0



Arquivos salvos: dataset_limpo.csv e dataset_limpo.pkl


## Nova analise de nulos apos a limpeza

Agora que removemos as colunas 100% nulas e as constantes, refazemos a analise em `df_limpo`. Esta etapa mostra os nulos parciais que continuam no dataset e prepara a proxima decisao de limpeza.

In [7]:
nulos_apos_limpeza = df_limpo.isna().sum()
colunas_com_nulos = nulos_apos_limpeza[nulos_apos_limpeza > 0].sort_values(ascending=False)
relatorio_nulos_apos_limpeza = colunas_com_nulos.rename('nulos').to_frame()
relatorio_nulos_apos_limpeza['percentual'] = (
    relatorio_nulos_apos_limpeza['nulos'] / len(df_limpo) * 100
).round(2)

print(f'Linhas: {len(df_limpo):,} | colunas: {df_limpo.shape[1]:,}')
print(f'Colunas com ao menos um nulo: {len(colunas_com_nulos)} de {df_limpo.shape[1]}')
print(f'Colunas 100% nulas: {int(df_limpo.isna().all().sum())}')
print(f'Total de valores nulos restantes: {int(nulos_apos_limpeza.sum()):,}')
print(f'Linhas sem nulos: {int(df_limpo.notna().all(axis=1).sum()):,} de {len(df_limpo):,}')
print('Colunas com mais nulos:')
display(relatorio_nulos_apos_limpeza.head(30))

Linhas: 9,146 | colunas: 264
Colunas com ao menos um nulo: 94 de 264
Colunas 100% nulas: 0
Total de valores nulos restantes: 220,977
Linhas sem nulos: 0 de 9,146
Colunas com mais nulos:


,nulos,percentual
rd_race_time_penalty,8917,97.50
rd_race_time_penalty_millis,8917,97.50
race_scheduled_laps,8422,92.08
race_scheduled_distance,8422,92.08
race_free_practice_3_date,8251,90.21
race_time,8251,90.21
race_free_practice_2_date,8251,90.21
race_free_practice_2_time,8251,90.21
race_free_practice_1_date,8251,90.21
race_free_practice_1_time,8251,90.21
